#### Loading Files

In [0]:
# Checking Files in Volume Path
dbutils.fs.ls("/Volumes/atci_databricks_hackathon_2025/atci_clinical_transcript_ai_hackathon/raw_data/")

#### Installing Packages
**Pdfplumber package for reading PDF files**,

**Pytesseract package for reading Images (OCR)**

In [0]:
%pip install pdfplumber pytesseract pillow

In [0]:
# Restarting Python
dbutils.library.restartPython()

In [0]:
# Importing Packages
import pdfplumber
import pytesseract
from PIL import Image
import os
import pandas as pd
from datetime import datetime, timezone

#### Reading from Raw data files

In [0]:
# Source Folder Path
SRC_FOLDER = "/Volumes/atci_databricks_hackathon_2025/atci_clinical_transcript_ai_hackathon/raw_data/"
records = []

for file_name in os.listdir(SRC_FOLDER):
    if not file_name.lower().endswith('.pdf'):
        continue
    filepath = os.path.join(SRC_FOLDER, file_name)
    with pdfplumber.open(filepath) as pdf:
        for page_num, page in enumerate(pdf.pages,start=1):
            # Reading page
            text = page.extract_text()

            # If text is empty then perform OCR
            if not text:
                image = page.to_image(resolution=300).original
                text = pytesseract.image_to_string(image)
            
            # appending the extracted data & Audit fields like User Name and Loaded Time    
            records.append({
                "file_name": file_name,
                "page_number": page_num,
                "page_text": text.strip(),
                "update_datetime": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                "updated_by": spark.sql("SELECT current_user()").collect()[0][0]
            })
        print(f"Reading {file_name} is completed, Successfully")

In [0]:
print(records[0])

In [0]:
df = spark.createDataFrame(records)
df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("atci_databricks_hackathon_2025.atci_clinical_transcript_ai_hackathon.clinial_data_extracted_bronze")

In [0]:
# Checking record count in Bronze Table
tb = spark.sql("""
               select file_name, count(*) as no_of_records from atci_databricks_hackathon_2025.atci_clinical_transcript_ai_hackathon.clinial_data_extracted_bronze
               group by file_name""")
display(tb)

In [0]:
# Checking image type record is loaded in Bronze Table
image_record = spark.sql("""
               select * from atci_databricks_hackathon_2025.atci_clinical_transcript_ai_hackathon.clinial_data_extracted_bronze where file_name='variable_patients_batch_1.pdf' and page_number=8""")
display(image_record)

#### ****Successfully Loaded into Bronze layer****